“While SQL provides the historical metrics of drivers, constructors, and circuits in isolation, executives need actionable insights that account for trends, consistency, relative performance, and reliability across multiple seasons. These insights cannot be efficiently extracted using SQL alone because they require rolling calculations, multi-year aggregations, and comparative scoring that incorporate context and smoothing over time. The goal is to transform historical F1 race data into executive-ready insights to guide decisions on driver retention, constructor investment, and circuit prioritization.”

Got it. We’re framing this at a **production/portfolio-ready, executive-level analysis perspective**. SQL handled **raw aggregations and foundational metrics**, but Pandas is where you add **multi-season trends, comparative insights, and derived intelligence that SQL alone cannot easily express**, especially with rolling windows, consistency, and peer-relative metrics.

Here’s how we can define the **project problem and the executive-level questions** your Pandas analysis will answer:

---

### **Project Problem Statement**

“While SQL provides the historical metrics of drivers, constructors, and circuits in isolation, executives need actionable insights that account for trends, consistency, relative performance, and reliability across multiple seasons. These insights cannot be efficiently extracted using SQL alone because they require rolling calculations, multi-year aggregations, and comparative scoring that incorporate context and smoothing over time. The goal is to transform historical F1 race data into executive-ready insights to guide decisions on driver retention, constructor investment, and circuit prioritization.”

---

### **Executive-Level Questions to Answer in Pandas**

**1. Time-Based Performance Trends (Drivers & Constructors)**

* How have top drivers’ win rates, podium rates, and points per race evolved season-to-season?
* Which constructors show improvement or decline in team performance over rolling 3–5 year windows?
* Which circuits have become more or less challenging over time (DNF trends, win variability)?

**Why Pandas:** Rolling averages and multi-season trend lines are cumbersome in SQL. Pandas allows window-based smoothing, multi-group rolling metrics, and fast experimentation with different time windows.

---

**2. Comparative Performance Analysis**

* Which drivers consistently outperform their peers within the same season and across seasons?
* How do constructors compare when accounting for driver DNFs and point accumulation?
* Which circuits are the most reliable or risky relative to other circuits?

**Why Pandas:** Percentile ranks, season-over-season ranking, and comparative top-N selection are computationally trivial in Pandas but require nested subqueries and complex CTEs in SQL. Pandas allows a fluid, production-level approach for executive dashboards.

---

**3. Consistency & Reliability Metrics**

* Which drivers maintain stable performance across multiple seasons?
* Which constructors have stable points accumulation despite variable driver outcomes?
* Which circuits consistently produce high variability or failures?

**Why Pandas:** SQL cannot natively compute rolling standard deviations, stability scores, or multi-metric consistency indices efficiently. Pandas allows building **custom performance consistency and volatility metrics** for executive insights.

---

**4. Failure & Risk Insights**

* Which drivers or constructors are most affected by DNFs, and what is the nature of those failures (driver vs mechanical)?
* Which circuits produce the highest risk of DNFs or inconsistent outcomes?
* How do DNFs impact overall championship performance trends for drivers and teams?

**Why Pandas:** Calculating rolling DNF rates, correlating DNFs with points lost, or categorizing failures is cumbersome in SQL but straightforward in Pandas with `.groupby()`, `.rolling()`, and conditional logic. This produces **actionable insights for risk management and decision-making**.

---

**Summary:**

* **SQL:** foundational metrics, aggregations, and master table creation.
* **Pandas:** trend analysis, comparative insights, consistency and reliability scoring, percentile rankings, risk profiling.
* **Tableau:** executive storytelling, visualization, dashboards for these insights.

This framing clearly justifies every choice in your workflow and positions your analysis as **production-level, executive-ready, and decision-oriented**.

If you want, I can next **convert this into a concise “Pandas Analysis Problem & Questions” section** that would fit directly in your README or project documentation — the kind recruiters/readers immediately understand your reasoning and analytical depth.

Do you want me to do that?


Step 1: Data Preparation / Validation
Goal: Clean, join, and validate the dataset to ensure trustworthy metrics.

1. Inspect with intent (not just select *)
For each table, answer:
	•	What is the primary key?
	•	What columns are actually needed for analysis?
	•	What columns are dirty or irrelevant?
Example thinking:
	•	results → core fact table (KEEP most)
	•	drivers → only name + id (DROP rest)
	•	constructors → name + id
	•	races → raceId, year, round, circuitId
	•	circuits → name, location

2. Build clean CTEs (not raw copies)
Each CTE should:
	•	Select only required columns
	•	Rename for clarity
	•	Cast types if needed
If your CTE still looks like select *, you’re doing it wrong.

3. Define the grain explicitly (most important)
You said:
“one driver per race per season”
Correct. Lock this mentally:
Grain = one driver in one race
Everything you join must respect this.

4. Join order (this is where people mess up)
Correct flow:
	1	Start with results (fact table)
	2	Join drivers (driverId)
	3	Join constructors (constructorId)
	4	Join races (raceId)
	5	Join circuits (via race)
If you join in random order, you risk duplication.

5. Validate after join (mandatory)
After creating master table, check:
	•	Row count = same as results
	•	No duplicate (driverId, raceId)
	•	Nulls in critical columns?
If this fails → your joins are wrong.

	•	SQL / BigQuery:
	◦	Join results_df with drivers_df, constructors_df, races_df, circuits_df.
	◦	Check for missing DNFs, duplicate entries, and invalid points.
	◦	Create intermediate tables: driver_results, constructor_results, race_summary.
	◦	Use CTEs and window functions to compute cumulative stats per driver/constructor.
	•	Python / Pandas:
	◦	Validate data consistency: duplicated(), isnull(), merge checks.
	◦	Create helper columns: season, age, rookie flag, race order.
Deliverable: Clean, normalized tables ready for analysis.

Step 2: Driver Performance & True Skill
SQL / BigQuery:
	•	Compute win rate, podium rate, points per race, DNF-adjusted points.
	•	Use window functions: RANK() over season per driver, LAG() to compute performance deltas.
	•	Compute driver vs teammate delta: points_this_race - teammate_points_this_race.
Python / Pandas:
	•	Compute rolling averages, trends over seasons, rookie vs veteran comparisons.
	•	Normalize metrics across eras or points system changes.
Looker Studio:
	•	KPI tiles: win rate, podium %, teammate delta.
	•	Trend charts: career trajectory, rookie growth, top drivers over seasons.
Insights: Highlight overperformers, consistent drivers, and risk-prone drivers (high DNF).

Step 3: Team / Constructor Analysis
SQL / BigQuery:
	•	Aggregate points per constructor, points per driver, win contribution ratios.
	•	Identify dependency on top driver via SUM(points)/SUM(total_points).
	•	Analyze DNFs by constructor and circuit.
Python:
	•	Compute variance metrics: driver contribution distribution.
	•	Operational efficiency: average points lost per race due to DNF or pit stop delays.
Looker Studio:
	•	Stacked bar charts: driver contribution per team.
	•	Scatter plots: reliability vs points per team.
Insights: Identify operational risks, driver dependency, and efficiency.

Step 4: Circuit & Race Analysis
SQL / BigQuery:
	•	Compute circuit-wise average winner dominance, number of unique winners, race unpredictability.
	•	Use window functions for track-based rankings, e.g., most frequent podiums per driver.
Python:
	•	Track variance, historical trends, impact of track type (street vs permanent).
Looker Studio:
	•	Heatmaps: circuits vs driver success rate.
	•	Charts: most unpredictable tracks, DNF-prone circuits.
Insights: Recommend circuits for competitive parity or highlight marketing storylines.

Step 5: Career & Talent Trajectory
SQL / BigQuery:
	•	Compute career points per season, peak season, rolling 3-year averages.
Python / Pandas:
	•	Identify rookies outperforming historical trends.
	•	Visualize age-performance curves.
Looker Studio:
	•	Line charts: driver career trajectories, peak performance age.
Insights: Scout emerging talent and guide retention/contract strategy.

Step 6: Competitive Balance
SQL / BigQuery:
	•	Compute concentration metrics: Gini coefficient of points, top driver/constructor share.
Python:
	•	Cross-era normalization, trend analysis.
Looker Studio:
	•	KPI tiles: % points held by top 3 drivers/teams per season.
	•	Line charts: competitive balance over decades.
Insights: Suggest league health interventions, dominance monitoring.

Step 7: Documentation & Presentation
	•	Document methodology: tables, calculations, assumptions.
	•	Explain insights per metric: metric → business impact → executive recommendation.
	•	Slides: Visuals + concise insights, no raw queries.
	•	Appendix: SQL snippets, Python derivations, data checks.


26 march 2026

My base table is results. All joins must preserve its row count and grain.

1. Objective (always first)
Build a master dataset at driver-race level to enable performance, team, and circuit analysis.”

1. Objective (always first)
Table: results
Grain: one driver per race
Primary key: (raceId, driverId)
Purpose: core performance metrics (position, points, grid, status)

results is the fact table. It defines performance at driver-race level. All metrics (win rate, points, DNF, consistency) originate from this table.
positionText is inconsistent and not used for logic. Race completion status will be derived 
from statusId via the status table.

Status is race status: finished, and def.
“Race outcomes will be classified into Finished, Mechanical Failure, Driver Error, and Disqualification to enable reliability and risk analysis.”

DNF and outcome classification will be applied in the final joined table, after integrating status with results.

drivers table enriches performance data with identity and enables career/age-based analysis.



All joins will be performed using surrogate keys (driverId, constructorId, raceId). Text references like driverRef are excluded to avoid inconsistency.”
————————
Drivers to be cleaned, handle null values and select columns

drivers table enriches performance data with identity and enables career/age-based analysis

constructors table enables team-level performance and dependency analysis.

races table provides temporal and event context, enabling season-level and circuit-level performance insights.


circuits table provides geographic and track-level context for performance and competitiveness analysis.



SQL layer will produce clean, reusable base metrics. Pandas layer will focus on advanced analytics and insight generation. No duplication of logic across layers.

3. Join Strategy (this is critical)
Join drivers on driverId
Join constructors on constructorId
Join races on raceId
Join circuits via races.circuitId

Data Validation Checks
Row count (results): X  
Row count (final): X → PASS  
Duplicate (driverId, raceId): 0 → PASS  

Execution model:

Join Order:
1. results + drivers (driver identity)
2. + constructors (team context)
3. + races (time context)
4. + circuits (location context)
5. + status (outcome classification)

5. Assumptions / Decisions
DNF defined as status != finished”
“Only numeric positions considered”
“Dropped unnecessary columns to reduce noise”




In [1]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project = 'project-1-music-489315')
print("Connected")

/opt/homebrew/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Connected


In [4]:
results_master_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.results_master`").to_dataframe()
results_master_df.head()

/opt/homebrew/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,grid,position,points,laps,fastestLap,driverId,driver_name,dob,driver_nationality,constructorId,...,circuitId,circuit_name,circuit_city,circuit_country,lat,lng,alt,statusId,race_status,failure_type
0,17,<NA>,0.0,43,\N,785,Geoff Crossley,1921-05-11,British,126,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,7,is_dnf,Mechanical / reliability failure
1,11,<NA>,0.0,26,\N,589,Louis Chiron,1899-08-03,Monegasque,105,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,8,is_dnf,Mechanical / reliability failure
2,7,<NA>,0.0,8,\N,789,Eugène Martin,1915-03-24,French,154,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,51,is_dnf,Mechanical / reliability failure
3,18,<NA>,0.0,44,\N,747,David Murray,1909-12-28,British,105,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,5,is_dnf,Mechanical / reliability failure
4,12,<NA>,0.0,2,\N,790,Leslie Johnson,1912-03-22,British,151,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,126,is_dnf,Mechanical / reliability failure


In [12]:
results_master_df.columns

Index(['grid', 'position', 'points', 'laps', 'fastestLap', 'driverId',
       'driver_name', 'dob', 'driver_nationality', 'constructorId',
       'constructor_name', 'constructor_nationality', 'raceId', 'race_year',
       'race_round', 'race_name', 'race_date', 'circuitId', 'circuit_name',
       'circuit_city', 'circuit_country', 'lat', 'lng', 'alt', 'statusId',
       'race_status', 'failure_type'],
      dtype='object')

In [5]:
driver_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.driver_metrics`").to_dataframe()
driver_metrics_df.head()

/opt/homebrew/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,driver_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Manfred Winkelhock,55,2.0,0,0,45,0.04,0.00,0.00,0.82
1,Lewis Hamilton,356,4820.5,105,202,44,13.54,0.29,0.57,0.12
2,Stirling Moss,73,186.5,16,24,38,2.55,0.22,0.33,0.52
3,Gilles Villeneuve,68,107.0,6,13,44,1.57,0.09,0.19,0.65
4,Keke Rosberg,128,159.5,5,17,86,1.25,0.04,0.13,0.67


In [7]:
constructor_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.constructor_metrics`").to_dataframe()
constructor_metrics_df.head()

/opt/homebrew/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,constructor_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,RAM,71,0.00,0,0,57,0.00,0.00,0.00,0.80
1,BAR,236,227.00,0,15,157,0.96,0.00,0.06,0.67
2,Maserati,436,313.14,9,40,272,0.72,0.02,0.09,0.62
3,Talbot-Lago,82,25.00,0,2,42,0.30,0.00,0.02,0.51
4,Osella,252,5.00,0,0,223,0.02,0.00,0.00,0.88


In [8]:
circuit_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.circuit_metrics`").to_dataframe()
circuit_metrics_df.head()

/opt/homebrew/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,circuit_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Circuit de Spa-Francorchamps,1258,2591.50,57,172,689,2.06,0.05,0.14,0.55
1,Silverstone Circuit,1436,2798.56,59,178,800,1.95,0.04,0.12,0.56
2,Autódromo José Carlos Pace,937,2203.00,41,123,551,2.35,0.04,0.13,0.59
3,Suzuka Circuit,791,1923.00,34,102,436,2.43,0.04,0.13,0.55
4,Nürburgring,976,1337.00,41,124,581,1.37,0.04,0.13,0.60


In [11]:
"""
Step 1: Driver Insights (pandas)

Focus on trend and comparative analysis:

Performance over seasons
Group by driver_name + year
Compute rolling win_rate, podium_rate, points_per_race
Identify improvement or decline trends
Consistency & reliability
Variance/std of finishing positions
Rolling DNF rate
Teammate comparisons
Merge driver_metrics with constructors per season
Compare driver vs teammate within same team
Step 2: Constructor Insights
Team dominance over time
Total points per season
Win/podium trends
Reliability
Mechanical vs driver-related failures
DNF distribution per team
Driver contribution
% of points contributed by each driver to constructor
Step 3: Circuit Insights
Circuit-specific difficulty
Average DNF per circuit
Average points per driver
Win rates per circuit
Location trends
Are certain drivers better in certain continents or climates?
"""

'\nStep 1: Driver Insights (pandas)\n\nFocus on trend and comparative analysis:\n\nPerformance over seasons\nGroup by driver_name + year\nCompute rolling win_rate, podium_rate, points_per_race\nIdentify improvement or decline trends\nConsistency & reliability\nVariance/std of finishing positions\nRolling DNF rate\nTeammate comparisons\nMerge driver_metrics with constructors per season\nCompare driver vs teammate within same team\nStep 2: Constructor Insights\nTeam dominance over time\nTotal points per season\nWin/podium trends\nReliability\nMechanical vs driver-related failures\nDNF distribution per team\nDriver contribution\n% of points contributed by each driver to constructor\nStep 3: Circuit Insights\nCircuit-specific difficulty\nAverage DNF per circuit\nAverage points per driver\nWin rates per circuit\nLocation trends\nAre certain drivers better in certain continents or climates?\n'

In [14]:
'''DRIVER TRENDS OVER SEASONS'''
driver_year = results_master_df.groupby(by=["driver_name", "race_year"]).agg(
    total_races = ("raceId", "count"),
    total_wins = ("position", lambda x: (x == 1).sum()),
    total_podiums = ("position", lambda x: (x <= 3).sum()),
    total_dnfs = ("race_status", lambda x: (x == "is_dnf").sum()),
    total_points = ("points", "sum")
).reset_index()

In [15]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points
0,Adolf Brudes,1952,1,0,0,1,0.0
1,Adolfo Cruz,1953,1,0,0,1,0.0
2,Adrian Sutil,2007,17,0,0,10,1.0
3,Adrian Sutil,2008,18,0,0,14,0.0
4,Adrian Sutil,2009,17,0,0,11,5.0


In [16]:
'''Per year rates'''
driver_year["win_rate"] = driver_year["total_wins"] / driver_year["total_races"]
driver_year["podium_rate"] = driver_year["total_podiums"] / driver_year["total_races"]
driver_year["dnf_rate"] = driver_year["total_dnfs"] / driver_year["total_races"]
driver_year["points_per_race"] = driver_year["total_points"] / driver_year["total_races"]

In [17]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.588235,0.058824
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.777778,0.0
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.647059,0.294118


In [19]:
'''Rolling metrics'''
driver_year = driver_year.sort_values(["driver_name", "race_year"])
driver_year["rolling_win_rate"] = driver_year.groupby("driver_name")["win_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_points"] = driver_year.groupby("driver_name")["points_per_race"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_podium_rate"] = driver_year.groupby("driver_name")["podium_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_dnf_rate"] = driver_year.groupby("driver_name")["dnf_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)

In [20]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,1.000000
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,1.000000
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.588235,0.058824,0.0,0.058824,0.0,0.588235
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.777778,0.0,0.0,0.029412,0.0,0.683007
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.647059,0.294118,0.0,0.117647,0.0,0.671024


In [22]:
'''CONSTRUCTOR TRENDS OVER SEASONS'''

constructor_year = results_master_df.groupby(["constructor_name", "race_year"]).agg(
    total_races = ("raceId", "count"),
    total_wins = ("position", lambda x : (x == 1).sum()),
    total_podiums = ("position", lambda x: (x <=3).sum()),
    total_dnfs = ("race_status", lambda x: (x == "is_dnf").sum()),
    total_points = ("points", "sum")
)

In [23]:
constructor_year.head()

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   

                            total_dnfs  total_points  
constructor_name race_year                            
AFM              1952                2           0.0  
                 1953                3           0.0  
AGS              1986                2           0.0  
                 1987                5           1.0  
                 1988               13           0.0

In [26]:
'''Constructor Per Year Metrics'''
constructor_year["win_rate"] = constructor_year["total_wins"] / constructor_year["total_races"]
constructor_year["podium_rate"] = constructor_year["total_podiums"] / constructor_year["total_races"]
constructor_year["dnf_rate"] = constructor_year["total_dnfs"] / constructor_year["total_races"]
constructor_year["points_per_race"] = constructor_year["total_points"] / constructor_year["total_races"]

In [27]:
'''Rolling Metrics for Constructors'''
constructor_year = constructor_year.sort_values(["constructor_name", "race_year"])
constructor_year["rolling_win_rate"] = constructor_year.groupby("constructor_name")["win_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_podium_rate"] = constructor_year.groupby("constructor_name")["podium_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_dnf_rate"] = constructor_year.groupby("constructor_name")["dnf_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_points"] = constructor_year.groupby("constructor_name")["points_per_race"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)

In [28]:
constructor_year.head(10)

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   
                 1989                31           0              0   
                 1990                32           0              0   
                 1991                28           0              0   
ATS              1963                17           0              0   
                 1978                32           0              0   

                            total_dnfs  total_points  win_rate  podium_rate  \
constructor_name race_year                                                    
AFM              1952                2           0.0       0.0          0.0   
                 1953                3           0.0       0.0          0.0   
AGS              1986                2           0.0       0.0          0.0   
                 1987                5           1.0       0.0          0.0   
                 1988               13           0.0       0.0          0.0   
                 1989               31           1.0       0.0          0.0   
                 1990               30           0.0       0.0          0.0   
                 1991               27           0.0       0.0          0.0   
ATS              1963               15           0.0       0.0          0.0   
                 1978               24           0.0       0.0          0.0   

                            dnf_rate  points_per_race  rolling_win_rate  \
constructor_name race_year                                                
AFM              1952       0.666667              0.0               0.0   
                 1953           0.75              0.0               0.0   
AGS              1986            1.0              0.0               0.0   
                 1987       0.357143         0.071429               0.0   
                 1988         0.8125              0.0               0.0   
                 1989            1.0         0.032258               0.0   
                 1990         0.9375              0.0               0.0   
                 1991       0.964286              0.0               0.0   
ATS              1963       0.882353              0.0               0.0   
                 1978           0.75              0.0               0.0   

                            rolling_podium_rate  rolling_dnf_rate  \
constructor_name race_year                                          
AFM              1952                       0.0          0.666667   
                 1953                       0.0          0.708333   
AGS              1986                       0.0          1.000000   
                 1987                       0.0          0.678571   
                 1988                       0.0          0.723214   
                 1989                       0.0          0.792411   
                 1990                       0.0          0.821429   
                 1991                       0.0          0.814286   
ATS              1963                       0.0          0.882353   
                 1978                       0.0          0.816176   

                            rolling_points  
constructor_name race_year                  
AFM              1952             0.000000  
                 1953             0.000000  
AGS              1986             0.000000  
                 1987             0.035714  
                 1988             0.023810  
                 1989             0.025922  
                 1990             0.020737  
                 1991             0.020737  
ATS              1963             0.000000  
             

In [31]:
# rounding off to 2 decimals
driver_year[["win_rate", "podium_rate", "dnf_rate",	"points_per_race", "rolling_win_rate", "rolling_points", "rolling_podium_rate", "rolling_dnf_rate"]] = driver_year[["win_rate", "podium_rate", "dnf_rate",	"points_per_race", "rolling_win_rate", "rolling_points", "rolling_podium_rate", "rolling_dnf_rate"]].round(2)
constructor_year[["win_rate",	"podium_rate",	"dnf_rate",	"points_per_race",	"rolling_win_rate",	"rolling_podium_rate",	"rolling_dnf_rate",	"rolling_points"]] = constructor_year[["win_rate",	"podium_rate",	"dnf_rate",	"points_per_race",	"rolling_win_rate",	"rolling_podium_rate",	"rolling_dnf_rate",	"rolling_points"]].round(2)

In [32]:
driver_year.head(10)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,1.00
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,1.00
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.59,0.06,0.0,0.06,0.0,0.59
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.78,0.0,0.0,0.03,0.0,0.68
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.65,0.29,0.0,0.12,0.0,0.67
5,Adrian Sutil,2010,19,0,0,7,47.0,0.0,0.0,0.37,2.47,0.0,0.71,0.0,0.60
6,Adrian Sutil,2011,19,0,0,12,42.0,0.0,0.0,0.63,2.21,0.0,1.01,0.0,0.60
7,Adrian Sutil,2013,19,0,0,10,29.0,0.0,0.0,0.53,1.53,0.0,1.30,0.0,0.59
8,Adrian Sutil,2014,19,0,0,17,0.0,0.0,0.0,0.89,0.0,0.0,1.30,0.0,0.61
9,Adrián Campos,1987,16,0,0,15,0.0,0.0,0.0,0.94,0.0,0.0,0.00,0.0,0.94


In [33]:
constructor_year.head(10)

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   
                 1989                31           0              0   
                 1990                32           0              0   
                 1991                28           0              0   
ATS              1963                17           0              0   
                 1978                32           0              0   

                            total_dnfs  total_points  win_rate  podium_rate  \
constructor_name race_year                                                    
AFM              1952                2           0.0       0.0          0.0   
                 1953                3           0.0       0.0          0.0   
AGS              1986                2           0.0       0.0          0.0   
                 1987                5           1.0       0.0          0.0   
                 1988               13           0.0       0.0          0.0   
                 1989               31           1.0       0.0          0.0   
                 1990               30           0.0       0.0          0.0   
                 1991               27           0.0       0.0          0.0   
ATS              1963               15           0.0       0.0          0.0   
                 1978               24           0.0       0.0          0.0   

                            dnf_rate  points_per_race  rolling_win_rate  \
constructor_name race_year                                                
AFM              1952           0.67              0.0               0.0   
                 1953           0.75              0.0               0.0   
AGS              1986            1.0              0.0               0.0   
                 1987           0.36             0.07               0.0   
                 1988           0.81              0.0               0.0   
                 1989            1.0             0.03               0.0   
                 1990           0.94              0.0               0.0   
                 1991           0.96              0.0               0.0   
ATS              1963           0.88              0.0               0.0   
                 1978           0.75              0.0               0.0   

                            rolling_podium_rate  rolling_dnf_rate  \
constructor_name race_year                                          
AFM              1952                       0.0              0.67   
                 1953                       0.0              0.71   
AGS              1986                       0.0              1.00   
                 1987                       0.0              0.68   
                 1988                       0.0              0.72   
                 1989                       0.0              0.79   
                 1990                       0.0              0.82   
                 1991                       0.0              0.81   
ATS              1963                       0.0              0.88   
                 1978                       0.0              0.82   

                            rolling_points  
constructor_name race_year                  
AFM              1952                 0.00  
                 1953                 0.00  
AGS              1986                 0.00  
                 1987                 0.04  
                 1988                 0.02  
                 1989                 0.03  
                 1990                 0.02  
                 1991                 0.02  
ATS              1963                 0.00  
             

In [42]:
print(driver_year["race_year"].dtype)

int64


In [43]:
print(driver_year["race_year"].head)

<bound method NDFrame.head of 0       1952
1       1953
2       2007
3       2008
4       2009
        ... 
3203    1991
3204    1992
3205    1993
3206    1994
3207    1956
Name: race_year, Length: 3208, dtype: int64>


In [44]:
driver_year["race_year"] = pd.to_numeric(driver_year["race_year"], errors = "coerce")

In [45]:
print(driver_year["race_year"].dtype)
print(driver_year["race_year"].head)

int64
<bound method NDFrame.head of 0       1952
1       1953
2       2007
3       2008
4       2009
        ... 
3203    1991
3204    1992
3205    1993
3206    1994
3207    1956
Name: race_year, Length: 3208, dtype: int64>


In [47]:
# top 20 drivers by win rate and points per race
recent = driver_year[driver_year["race_year"] >= 2015]
top = recent.sort_values("rolling_win_rate", ascending = False).head(15)

In [48]:
top

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
1887,Lewis Hamilton,2020,16,11,14,0,347.0,0.69,0.88,0.0,21.69,0.53,19.41,0.79,0.04
1885,Lewis Hamilton,2018,21,11,17,1,408.0,0.52,0.81,0.05,19.43,0.51,19.19,0.80,0.08
1888,Lewis Hamilton,2021,22,8,17,1,385.5,0.36,0.77,0.05,17.52,0.51,19.29,0.78,0.03
2097,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11
1886,Lewis Hamilton,2019,21,11,17,0,413.0,0.52,0.81,0.0,19.67,0.50,19.08,0.79,0.05
2096,Max Verstappen,2023,22,19,21,0,530.0,0.86,0.95,0.0,24.09,0.45,17.45,0.72,0.12
1889,Lewis Hamilton,2022,22,0,9,3,233.0,0.0,0.41,0.14,10.59,0.42,17.78,0.74,0.05
1884,Lewis Hamilton,2017,20,9,13,1,363.0,0.45,0.65,0.05,18.15,0.42,17.29,0.69,0.09
1883,Lewis Hamilton,2016,21,10,17,2,380.0,0.48,0.81,0.1,18.1,0.37,15.56,0.63,0.14
2875,Sebastian Vettel,2015,19,3,13,2,278.0,0.16,0.68,0.11,14.63,0.33,15.80,0.63,0.09


In [ ]:
latest = driver_year.sort_values("race_year").groupby("driver_name").tail(1) # sort by race year ascending, then group by driver name. now, the output is race_year, driver name. out of these, we need latest race year, so tail(1) gives us the last year a particular driver had a race
latest = latest.merge(driver_metrics_df[["driver_name", "total_races"]].rename(columns = {"total_races": "career_races"}), on="driver_name") # 
latest = latest[latest["career_races"] >= 50]
top = latest.sort_values("rolling_win_rate", ascending = False).head(10)

In [91]:
top.head(20)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate,career_races
153,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11,209
9,Jim Clark,1968,1,1,1,0,9.0,1.0,1.0,0.0,9.0,0.49,4.73,0.53,0.47,73
0,Juan Fangio,1958,2,0,0,0,7.0,0.0,0.0,0.0,3.5,0.46,5.50,0.61,0.17,58
18,Jackie Stewart,1973,15,5,8,4,71.0,0.33,0.53,0.27,4.73,0.37,4.42,0.51,0.34,100
3,Stirling Moss,1961,9,2,2,5,21.0,0.22,0.22,0.56,2.33,0.32,3.18,0.39,0.48,73
72,Alain Prost,1993,16,7,12,4,99.0,0.44,0.75,0.25,6.19,0.29,4.93,0.64,0.27,202
80,Ayrton Senna,1994,3,0,0,3,0.0,0.0,0.0,1.0,0.0,0.26,3.71,0.46,0.49,162
88,Nigel Mansell,1995,2,0,0,1,0.0,0.0,0.0,0.5,0.0,0.24,3.36,0.38,0.45,192
99,Mika Häkkinen,2001,17,2,3,8,37.0,0.12,0.18,0.47,2.18,0.24,4.00,0.46,0.36,165
145,Lewis Hamilton,2024,24,2,5,2,207.0,0.08,0.21,0.08,8.62,0.23,13.66,0.51,0.07,356
